In [ ]:
import neo_api_client

In [ ]:
from neo_api_client import NeoAPI
import pyotp
import time

# -----------------------------
# CONFIGURATION
# -----------------------------

CONSUMER_KEY = "a639e946-b78e-4e43-a071-613b27ea82c8"   # from Kotak Neo Trade API dashboard
MOBILE_NUMBER = "+918806160768"      # with country code
UCC = "XVV28"               # your client code
MPIN = "123457"                       # your Neo account MPIN
TOTP_SECRET = "YOUR_TOTP_SECRET"    # secret key from authenticator QR


In [ ]:
from neo_api_client import NeoAPI

# ---------------------------------
# USER DETAILS
# ---------------------------------

CONSUMER_KEY = "YOUR_CONSUMER_KEY"   # from Kotak Neo Trade API dashboard
MOBILE_NUMBER = "+91XXXXXXXXXX"      # with country code
UCC = "YOUR_UCC"               # your client code
MPIN = "YOUR_MPIN"                       # your Neo account MPIN

# ---------------------------------
# CREATE CLIENT
# ---------------------------------

client = NeoAPI(
    environment="prod",
    access_token=None,
    neo_fin_key=None,
    consumer_key=CONSUMER_KEY
)

# ---------------------------------
# TAKE TOTP INPUT FROM USER
# ---------------------------------

totp = input("Enter TOTP from Authenticator: ")

# Step 1: TOTP Login
login_response = client.totp_login(
    mobile_number=MOBILE_NUMBER,
    ucc=UCC,
    totp=totp
)

print("Login Response:", login_response)

# Step 2: Validate MPIN
session_response = client.totp_validate(mpin=MPIN)

print("Session Response:", session_response)

print("Login Successful")

In [ ]:
order = client.place_order(
    exchange_segment="nse_cm",
    product="MIS",
    price="0",
    order_type="MKT",
    quantity="1",
    validity="DAY",
    trading_symbol="RELIANCE-EQ",
    transaction_type="B",
    amo="NO"
)

print(order)

In [ ]:
order = client.place_order(
    exchange_segment="nse_cm",
    product="MIS",
    price="2500",
    order_type="L",
    quantity="1",
    validity="DAY",
    trading_symbol="RELIANCE-EQ",
    transaction_type="B",
    amo="NO"
)

print(order)

In [ ]:
order = client.place_order(
    exchange_segment="nse_cm",
    product="MIS",
    price="2500",
    order_type="SL-M",
    quantity="1",
    validity="DAY",
    trading_symbol="RELIANCE-EQ",
    transaction_type="B",
    trigger_price="2498",
    amo="NO"
)

print(order)

In [ ]:
order = client.place_order(
    exchange_segment="nse_fo",
    product="MIS",
    price="0",
    order_type="MKT",
    quantity="65",
    validity="DAY",
    trading_symbol="NIFTY26SEP13500PE",
    transaction_type="B",
    amo="YES"
)

print(order)

In [ ]:
order = client.place_order(
    exchange_segment="nse_fo",
    product="MIS",
    price="320",
    order_type="L",
    quantity="65",
    validity="DAY",
    trading_symbol="NIFTY2631024500CE",
    transaction_type="S",
    amo="NO"
)

print(order)

In [ ]:
client.modify_order(order_id = "260304000597143", price = "340.0", quantity = "65", disclosed_quantity = "0", trigger_price = "0", validity = "DAY", order_type='L')






In [ ]:
import time

price = 340.0

for i in range(50):

    price += 0.5

    client.modify_order(
        order_id="260304000598135",
        price=str(price),
        quantity="65",
        disclosed_quantity="0",
        trigger_price="0",
        validity="DAY",
        order_type="L"
    )

    print("Iteration:", i+1, "New Price:", price)



In [ ]:
import time

symbol = "NIFTY2631024500CE"
price = "320"
qty = "65"

# warm-up (optional)
client.order_report()

start = time.perf_counter()

order = client.place_order(
    exchange_segment="nse_fo",
    product="MIS",
    price=price,
    order_type="L",
    quantity=qty,
    validity="DAY",
    trading_symbol=symbol,
    transaction_type="B",
    amo="YES"
)

latency_ms = (time.perf_counter() - start) * 1000

print(f"Latency: {latency_ms:.2f} ms")

In [ ]:
import time

symbol = "NIFTY2631024500CE"
qty = "65"

# -------------------------
# 1. PLACE INITIAL ORDER
# -------------------------

initial_price = "300"

order = client.place_order(
    exchange_segment="nse_fo",
    product="MIS",
    price=initial_price,
    order_type="L",
    quantity=qty,
    validity="DAY",
    trading_symbol=symbol,
    transaction_type="S",
    amo="YES"
)

print("Initial Order Response:", order)

# -------------------------
# 2. STORE ORDER ID
# -------------------------

order_id = order["nOrdNo"]   # adjust key if API response differs
print("Stored Order ID:", order_id)

# -------------------------
# 3. TRAILING DEMONSTRATION
# -------------------------

limit_price = initial_price

for i in range(10):

    limit_price += 2

    modify = client.modify_order(
        order_id=order_id,
        price=limit_price,
        quantity=qty,
        trigger_price="0",
        validity="DAY",
        order_type="L"
    )

    print("Iteration:", i+1, "New Limit Price:", limit_price)

    time.sleep(1)

In [ ]:
import pandas as pd

url = client.scrip_master(exchange_segment="nse_fo")

df = pd.read_csv(url)

print(df.head())

In [ ]:
df.info

In [ ]:
nifty_df = df[df["pSymbolName"] == "NIFTY"]

print(nifty_df.head())

In [ ]:
march_options = nifty_df[nifty_df["pTrdSymbol"].str.contains("MAR")]

print(march_options[["pTrdSymbol"]].tail())

In [ ]:
nearest_expiry = sorted(df["pExpiryDate"].unique())[0]

weekly_df = df[df["pExpiryDate"] == nearest_expiry]

In [ ]:
weekly_df